In [1]:
import nltk
import numpy as np
import string
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

## Download NLTK's resources if they are not present in local machine.

In [2]:
def init_nltk_resources():
    nltk.download('stopwords')
    nltk.download('punkt')
    nltk.download('wordnet')
    nltk.download('omw-1.4')

In [3]:
init_nltk_resources()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ameli\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [4]:
def read_data(path):
	try:
		# use Python to read in our text file
		file = open(path, encoding='utf-8')

        # read entire file at one go
		doc = file.read()

		file.close()

		return doc
	except FileNotFoundError:
		return None

## Preprocess

#### break into sentences.
#### for each sentence:

##### 1. punctuation-removal

##### 2. stopwords-removal

##### 3. lemmatization


In [5]:
def preprocessing(doc):
    # use the English stopwords provided by NLTK
    stop_words = stopwords.words('english')

    # get a Stemmer to convert words to root forms
    stemmer = nltk.stem.PorterStemmer()
    sentences = nltk.tokenize.sent_tokenize(doc)

    # treat each sentence as a document
    docs = []

    # map every punctuation found in string.punctuation
    # to an empty character (essentially removing them
    # from our corpus)
    punc = str.maketrans('','', string.punctuation)

    # perform stemming on each word, and convert each word to
    # lowercase (so that words 'Bad', 'BAD', 'bad' are all treated
    # the same)
    for sent in sentences:
        sent_no_punc = sent.translate(punc)
        words_stemmed = [stemmer.stem(w)
            for w in sent_no_punc.lower().split()
                if w not in stop_words]
        docs += [' '.join(words_stemmed)]

    # return an array of "documents" - the "documents" are
    # actually sentences in our original document
    return docs

In [6]:
'''
Perform text featurization. Here, we use TF-IDF to convert text as
feature-vectors. After that, we convert the generated feature-vectors
into a Pandas dataframe to facilitate our computations later.
'''
def gen_features_as_dataframe(docs):
    # get a TF-IDF object from sklearn
    tfidf = TfidfVectorizer()

    # convert to NumPy array
    tfidf_vecs = tfidf.fit_transform(docs).toarray()

    # index for our dataframe
    df_index = ['doc'+str(i) for i in range(len(docs))]

    # columns for our dataframe
    df_columns = tfidf.get_feature_names_out()

    return pd.DataFrame(data=tfidf_vecs,
        index=df_index, columns=df_columns)


In [7]:
'''
Get top N documents with the largest TF-IDF values.
'''
def get_top_tfidf_sums(df, n):
    # sum up the TF-IDF values for each document
    # axis=1 because the columns in our dataframe
    # are collapsed to give us a total value
    tfidf_sum_by_docs = df.sum(axis=1)
    print(f"tfidf_sum_by_docs =\n{tfidf_sum_by_docs}\n")


    # sort the computed sums in descending order
    # as we want the "documents" with the highest
    # TF-IDF values
    # sorted_series = tfidf_sum_by_docs.sort_values(ascending=False)
    sorted_series = tfidf_sum_by_docs.sort_values(ascending=False)

    # only returns the top N "documents"
    return sorted_series.head(n)

In [8]:
'''
Print out the sentences in the original document with the
highest TF-IDF values.
'''
def print_summary(top_series, doc):
    # remove the prefix 'doc' from indexes such as
    # [doc10, doc24, doc11, ..., doc22]
    num_only = [int(x[3:]) for x in top_series.index]

    # we now have the top 10 "documents", but we want
    # our summary to show them as sentences as appear
    # in the same order as the original document
    sorted_num_only = sorted(num_only)
    print(f"sorted_num_only = {sorted_num_only}\n")

    # reference back to our original document
    sentences = nltk.tokenize.sent_tokenize(doc)

    # print out sentences, with top TF-IDF values,
    # in the order that they appear in the original
    # document
    for i in sorted_num_only:
        # due to zero-based indicing, our i starts from 0.
        # hence, our line numbering should add 1.
        print('[Line {}] - {}\n'.format(i+1, sentences[i]))


In [9]:
# reading in our dataset
doc = read_data('space_invaders.txt')

# perform data processing
docs = preprocessing(doc)
print(f"\nAfter preprocessing: \n{docs}\n")


After preprocessing: 
['space invad fix shooter player control laser cannon move horizont across bottom screen fire descend alien', 'aim defeat five row eleven aliens—although version featur differ numbers—that move horizont back forth across screen advanc toward bottom screen', 'player laser cannon partial protect sever stationari defens bunkers—th number also vari version—that gradual destroy top bottom blast either alien player', 'player defeat alien earn point shoot laser cannon', 'alien defeat alien movement game music speed', 'defeat alien onscreen bring anoth wave difficult loop continu endlessli', 'special mysteri ship occasion move across top screen award bonu point destroy', 'alien attempt destroy player cannon fire approach bottom screen', 'reach bottom alien invas declar success game end tragic otherwis end gener player last cannon destroy enemi projectil', 'space invad creat japanes design tomohiro nishikado spent year design game develop necessari hardwar produc', 'game 

In [10]:
# generate TF-IDF features
df = gen_features_as_dataframe(docs)
print(f"Generated TF-IDF features = \n{df}\n")

Generated TF-IDF features = 
         100yen      1972      1974      1975      1978     1979      8080  \
doc0   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc1   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc2   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc3   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc4   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc5   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc6   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc7   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc8   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc9   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
doc10  0.000000  0.233159  0.000000  0.000000  0.000000  0.00000  0.000000   
doc11  0.000000  0.000000  0.000000

In [11]:
# get top 10 documents by computed TF-IDF values
top_series = get_top_tfidf_sums(df, 10)
print(f"Top 10 TF-IDF sums:\n{top_series}\n")

tfidf_sum_by_docs =
doc0     3.912255
doc1     4.525808
doc2     4.628859
doc3     2.781502
doc4     2.376074
doc5     3.123966
doc6     3.430478
doc7     2.952512
doc8     3.908374
doc9     3.604040
doc10    4.821202
doc11    2.570495
doc12    3.536538
doc13    2.616532
doc14    3.561863
doc15    2.917672
doc16    3.280913
doc17    2.799709
doc18    3.834832
doc19    3.574759
doc20    2.171319
doc21    3.419013
doc22    4.235573
doc23    2.803102
doc24    4.816107
doc25    4.300563
doc26    4.149037
doc27    3.564487
doc28    5.007711
doc29    3.649207
doc30    3.424394
doc31    3.138539
doc32    4.040074
doc33    1.655060
doc34    1.000000
doc35    2.646406
doc36    4.399896
doc37    4.715729
doc38    2.615499
doc39    4.402610
doc40    3.408534
doc41    3.241754
doc42    3.156477
doc43    3.595558
doc44    3.241754
doc45    3.591621
doc46    4.154909
doc47    3.179854
doc48    5.948904
doc49    4.171724
doc50    2.587288
doc51    3.668989
dtype: float64

Top 10 TF-IDF sums:
doc48   

In [12]:
# show extracted sentences as summary
print_summary(top_series, doc);

sorted_num_only = [1, 2, 10, 24, 25, 28, 36, 37, 39, 48]

[Line 2] - The aim is to defeat five rows of eleven aliens—although some versions feature different numbers—that move horizontally back and forth across the screen as they advance toward the bottom of the screen.

[Line 3] - The player's laser cannon is partially protected by several stationary defense bunkers—the number also varies by version—that are gradually destroyed from the top and bottom by blasts from either the aliens or the player.

[Line 11] - The game's inspiration is reported to have come from varying sources, including an adaptation of the mechanical game Space Monsters released by Taito in 1972, and a dream about Japanese school children who are waiting for Santa Claus when they are attacked by invading aliens.

[Line 25] - The game uses an Intel 8080 central processing unit (CPU), displays raster graphics on a CRT monitor using a bitmapped framebuffer, and uses monaural sound hosted by a combination of analog ci